# Pandas ile Veri Temizleme (E-Ticaret Satış Verisi ile)

Bu notebook, `messy_ecommerce_sales_data.csv` dosyasındaki **gerçek veri kalitesi sorunlarını** kullanarak pandas ile veri temizleme sürecini adım adım gösterir.

Veri setinde tespit edilen gerçek sorunlar:
- **Sütun adlarında boşluklar** (örn. ` Customer_Name`, ` Category`)
- **Eksik değerler** (`Category`, `Quantity`, `Price` ve `Total` sütunlarında NaN değerler)
- **Yinelenen (Duplicate) satırlar**
- **Metin olarak saklanan ve bozuk veri içeren sayısal sütunlar** (`Price` sütununda harf/metin içeren hatalı girdiler)
- **Metin olarak saklanan tarih sütunu** (`Order_Date` datetime değil, string)
- **Eksik veya hatalı hesaplanmış toplam tutarlar** (`Total` sütunu)
- **Kategori ve ödeme yöntemlerinde yazım/boşluk tutarsızlıkları**

**İçerik:**
1. Veriyi Yükleme ve İlk Bakış
2. Genel Veri Sağlığı Kontrolü
3. Eksik Değerlerin Tespiti ve Giderilmesi
4. Yinelenen (Duplicate) Kayıt Kontrolü
5. Veri Tiplerinin Düzeltilmesi (Tarih ve Sayısal Sütunlar)
6. Mantıksal Hata / Aykırı Değer Tespiti ve Düzeltilmesi (Toplam Tutar Hesaplama)
7. Kategori Tutarlılığı Kontrolü
8. İndeks Düzenleme
9. Temizlenmiş Veriyi Kaydetme
10. Uçtan Uca Temizleme Fonksiyonu

## 1. Veriyi Yükleme ve İlk Bakış

CSV dosyasını okuyoruz ve verinin genel yapısını inceliyoruz.

In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv('messy_ecommerce_sales_data.csv')
print("Boyut:", df_ham.shape)
df.head()

Boyut: (103, 11)


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


## 2. Genel Veri Sağlığı Kontrolü

Temizliğe başlamadan önce verinin genel durumunu anlamak gerekir:

- `df.info()` → veri tipleri ve boş değer sayıları
- `df.isnull().sum()` → sütun bazında eksik değer sayısı
- `df.duplicated().sum()` → tekrar eden satır sayısı

In [17]:
# Sütun adlarındaki baştaki/sondaki boşlukları temizleme (.str.strip() kullanımı)
df.columns = df.columns.str.strip()
print("Düzenlenmiş Sütunlar:", df.columns.tolist())

df.info()

Düzenlenmiş Sütunlar: ['ID', 'Customer_Name', 'Order_ID', 'Order_Date', 'Product', 'Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ID              103 non-null    int64  
 1   Customer_Name   103 non-null    object 
 2   Order_ID        103 non-null    object 
 3   Order_Date      103 non-null    object 
 4   Product         103 non-null    object 
 5   Category        95 non-null     object 
 6   Quantity        98 non-null     object 
 7   Price           98 non-null     object 
 8   Payment_Method  103 non-null    object 
 9   Status          103 non-null    object 
 10  Total           89 non-null     float64
dtypes: float64(1), int64(1), object(9)
memory usage: 9.0+ KB


In [18]:
print("Sütun bazında eksik değer sayısı:")
print(df.isnull().sum())

print("\nToplam tekrar eden satır sayısı:", df.duplicated().sum())

Sütun bazında eksik değer sayısı:
ID                 0
Customer_Name      0
Order_ID           0
Order_Date         0
Product            0
Category           8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64

Toplam tekrar eden satır sayısı: 1


## 3. Eksik Değerlerin Tespiti ve Giderilmesi

Eksik kategorileri `"Diğer"` veya `"Bilinmiyor"` ile, eksik adetleri ise ortalama/medyan ile doldurabiliriz.

Burada iki mantıklı seçenek var:
1. **Satırları silmek** (eğer hangi mağazada olduğu analiz için kritikse ve tahmin edilemiyorsa)
2. **`"Bilinmiyor"` gibi bir yer tutucu ile doldurmak** (eğer diğer sütunlar hâlâ analiz için değerliyse, örn. ürün bazlı analiz yapılacaksa satırı tamamen kaybetmek istemeyiz)

Biz burada veriyi kaybetmemek için ikinci yolu seçiyoruz ve bu kararı **açıkça belirtiyoruz**.

In [27]:
#'Price' sütunundaki eksik değerleri kontrol ediyoruz
eksik_degerler = df[df['Price'].isnull()]
print(f"Price sütununda eksik olan satır sayısı: {len(eksik_degerler)}")
eksik_degerler.head()

Price sütununda eksik olan satır sayısı: 5


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
16,116,Customer_116,ORD-63660,10/30/2025,Microwave,Home,4,NaN,Cash on Delivery,Processing,NaN
24,124,Customer_124,ORD-46136,5/31/2025,Comics,Books,5,NaN,PayPal,Cancelled,NaN
30,130,Customer_130,ORD-34007,8/15/2025,Tennis Racket,Sports,2,NaN,Bank Transfer,Processing,NaN
56,156,Customer_156,ORD-34679,11/28/2024,Jeans,Clothing,2,NaN,Bank Transfer,Shipped,NaN
83,183,Customer_183,ORD-20916,3/10/2025,Shoes,Clothing,5,NaN,Cash on Delivery,Returned,NaN


In [38]:
df_temiz = df.copy()

# Kategori eksiklerini doldur
df_temiz['Category'] = df_temiz['Category'].fillna('Bilinmiyor')

# Adet (Quantity) sütununu sayısal tipe çevir (metin içerenler NaN olur). Eğer sütunda sayıya dönüştürülemeyecek bir değer (örneğin '4a' gibi bir metin) varsa, errors='coerce' parametresi sayesinde bu değerler NaN olarak işaretlenir.
df_temiz['Quantity'] = pd.to_numeric(df_temiz['Quantity'], errors='coerce')
# Adet (Quantity) eksiklerini ortalama ile doldur
df_temiz['Quantity'] = df_temiz['Quantity'].fillna(df_temiz['Quantity'].mean())

print("Eksik değerlerin giderilmesinden sonra kalan NaN sayıları:", df_temiz.isnull().sum())
df_temiz['Category'].value_counts()


Eksik değerlerin giderilmesinden sonra kalan NaN sayıları: ID                 0
Customer_Name      0
Order_ID           0
Order_Date         0
Product            0
Category           0
Quantity           0
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64


,count
Category,
Books,22
Home,20
Sports,16
Clothing,15
Electronics,11
Bilinmiyor,8
electronic,4
ELECTRONICS,3
electronics,3


## 4. Yinelenen (Duplicate) Kayıt Kontrolü

Veri setinde tespit edilen tekrar eden satırları temizliyoruz.

In [39]:
tekrar_sayisi = df_temiz.duplicated().sum()
print(f"Tekrar eden satır sayısı: {tekrar_sayisi}")

df_temiz = df_temiz.drop_duplicates().reset_index(drop=True)
print("Temizlik sonrası satır sayısı:", len(df_temiz))

Tekrar eden satır sayısı: 1
Temizlik sonrası satır sayısı: 102


## 5. Veri Tiplerinin Düzeltilmesi (Tarih ve Sayısal Sütunlar)

`Order_Date` sütununu datetime tipine, `Price` ve `Quantity` sütunlarını sayısal tiplere çeviriyoruz (Price içinde harf içeren hatalı değerler `NaN` olacaktır).

In [44]:
print("Dönüşümden önce tip:", df_temiz['Order_Date'].dtype)
print("Örnek değer:", df_temiz['Order_Date'].iloc[0])

df_temiz['Order_Date'] = pd.to_datetime(df_temiz['Order_Date'], errors='coerce')

print("\nDönüşümden sonra tip:", df_temiz['Order_Date'].dtype)
print("Çevrilemeyen (NaT) satır sayısı:", df_temiz['Order_Date'].isnull().sum())

# Artık tarihten yıl/ay gibi bilgiler çıkarabiliriz
df_temiz['islem_yili'] = df_temiz['Order_Date'].dt.year
df_temiz['islem_ayi'] = df_temiz['Order_Date'].dt.month
df_temiz[['Order_Date', 'islem_yili', 'islem_ayi']].head()

Dönüşümden önce tip: datetime64[ns]
Örnek değer: 2024-11-22 00:00:00

Dönüşümden sonra tip: datetime64[ns]
Çevrilemeyen (NaT) satır sayısı: 2


,Order_Date,islem_yili,islem_ayi
0,2024-11-22,2024.0,11.0
1,2025-07-05,2025.0,7.0
2,2024-12-23,2024.0,12.0
3,2025-03-19,2025.0,3.0
4,2025-10-20,2025.0,10.0


In [45]:
df_temiz[df_temiz['Order_Date'].isnull()]

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,islem_yili,islem_ayi
14,114,Customer_114,ORD-77417,NaT,Smartphone,Electronics,2.000000,413.13,Cash on Delivery,Shipped,826.260000,NaN,NaN
92,192,Customer_192,ORD-35144,NaT,Jacket,Clothing,2.917526,203.63,Credit Card,Returned,594.095773,NaN,NaN


## 6. Mantıksal Hata / Aykırı Değer Tespiti ve Düzeltilmesi (Toplam Tutar)

`Total` (Toplam tutar) sütunu `Quantity * Price` değerine eşit olmalıdır. Eksik veya hatalı toplamları bu formülle yeniden hesaplıyoruz.

In [43]:
# Toplam tutarı mantıksal olarak yeniden hesapla
df_temiz['Total'] = df_temiz['Quantity'] * df_temiz['Price']
print("Toplam tutarlar Quantity * Price formülüyle yeniden hesaplandı.")
df_temiz[['Quantity', 'Price', 'Total']].head()

Toplam tutarlar Quantity * Price formülüyle yeniden hesaplandı.


,Quantity,Price,Total
0,3.0,38.00,114.00
1,2.0,538.35,1076.70
2,1.0,389.05,389.05
3,5.0,233.92,1169.60
4,1.0,552.51,552.51


In [48]:
df_temiz['calculated_total'] = df_temiz['Quantity'] * df_temiz['Price']

tutarsizliklar = df_temiz[df_temiz['Total'] != df_temiz['calculated_total']]

print(f"'Total' ve 'Quantity * Price' arasındaki tutarsız satır sayısı: {len(tutarsizliklar)}")
display(tutarsizliklar[['Product', 'Quantity', 'Price', 'Total', 'calculated_total']])

'Total' ve 'Quantity * Price' arasındaki tutarsız satır sayısı: 0


,Product,Quantity,Price,Total,calculated_total


## 7. Kategori Tutarlılığı Kontrolü

Kategorik sütunlarda boşluk ve büyük/küçük harf tutarsızlıklarını gideriyoruz.

In [50]:
df_temiz['Category'] = df_temiz['Category'].str.strip().str.title()
df_temiz['Payment_Method'] = df_temiz['Payment_Method'].str.strip()

print("Kategoriler:", df_temiz['Category'].unique())
print("Ödeme Yöntemleri:", df_temiz['Payment_Method'].unique())

Kategoriler: ['Home' 'Electronics' 'Sports' 'Books' 'Clothing' 'Electronic'
 'Bilinmiyor']
Ödeme Yöntemleri: ['Cash on Delivery' 'PayPal' 'Bank Transfer' 'Credit Card']


## 8. İndeks Düzenleme

İşlemler sonrasında bozulabilecek indeksleri sıfırdan düzenliyoruz.

In [53]:
df_temiz = df_temiz.reset_index(drop=True)
print("İndeks sıfırlandı. İlk 5 satır:")
df_temiz.head()

İndeks sıfırlandı. İlk 5 satır:


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,islem_yili,islem_ayi,calculated_total
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3.0,38.00,Cash on Delivery,Shipped,114.00,2024.0,11.0,114.00
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2.0,538.35,PayPal,Processing,1076.70,2025.0,7.0,1076.70
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1.0,389.05,PayPal,Delivered,389.05,2024.0,12.0,389.05
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5.0,233.92,PayPal,Processing,1169.60,2025.0,3.0,1169.60
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1.0,552.51,Cash on Delivery,Processing,552.51,2025.0,10.0,552.51


## 9. Temizlenmiş Veriyi Kaydetme

Temizlenmiş veriyi yeni bir CSV dosyası olarak kaydediyoruz.

In [54]:
df_temiz.to_csv('messy_ecommerce_sales_data_temiz.csv', index=False)
print("Temizlenmiş veri 'messy_ecommerce_sales_data_temiz.csv' olarak kaydedildi.")
df_temiz.info()

Temizlenmiş veri 'messy_ecommerce_sales_data_temiz.csv' olarak kaydedildi.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ID                102 non-null    int64         
 1   Customer_Name     102 non-null    object        
 2   Order_ID          102 non-null    object        
 3   Order_Date        100 non-null    datetime64[ns]
 4   Product           102 non-null    object        
 5   Category          102 non-null    object        
 6   Quantity          102 non-null    float64       
 7   Price             102 non-null    float64       
 8   Payment_Method    102 non-null    object        
 9   Status            102 non-null    object        
 10  Total             102 non-null    float64       
 11  islem_yili        100 non-null    float64       
 12  islem_ayi         100 non-null    float64       
 13  calcu

## 10. Uçtan Uca Temizleme Fonksiyonu

Tüm bu adımları tek bir fonksiyonda toplayarak yeniden üretilebilir (reproducible) hale getirelim.

In [62]:
def veriyi_temizle(df_ham):
    """
    Ham e-ticaret DataFrame'ini alır, temel temizleme adımlarını uygular
    ve temizlenmiş DataFrame'i döndürür.
    """
    df = df_ham.copy()

    # Sütun adlarındaki baştaki/sondaki boşlukları temizleme
    df.columns = df.columns.str.strip()

    # Category eksiklerini doldur
    df['Category'] = df['Category'].fillna('Bilinmiyor').str.strip().str.title()

    # Quantity sütununu sayısal tipe çevir, hataları NaN yap ve NaN'ları medyan ile doldur
    df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
    df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

    # Price sütununu sayısal tipe çevir, hataları NaN yap ve NaN'ları medyan ile doldur
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    df['Price'] = df['Price'].fillna(df['Price'].median())

    # Order_Date sütununu datetime'a çevir
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

    # Toplam tutarı Quantity * Price formülüyle yeniden hesapla
    df['Total'] = df['Quantity'] * df['Price']

    # Payment_Method boşluklarını temizle
    df['Payment_Method'] = df['Payment_Method'].str.strip()

    # Tekrar eden kayıtları temizle (tüm sütunları kontrol ederek, keep='first')
    df = df.drop_duplicates().reset_index(drop=True)

    # Ekstra sütunları (islem_yili, islem_ayi) ekle (eğer isteniyorsa)
    df['islem_yili'] = df['Order_Date'].dt.year
    df['islem_ayi'] = df['Order_Date'].dt.month

    return df

# Fonksiyonu ham veri üzerinde test edelim
df_sonuc = veriyi_temizle(df_ham)

print("Temizleme sonrası kontrol:")
print(f"- Eksik 'Category' sayısı: {df_sonuc['Category'].isnull().sum()}")
print(f"- Eksik 'Quantity' sayısı: {df_sonuc['Quantity'].isnull().sum()}")
print(f"- Eksik 'Price' sayısı: {df_sonuc['Price'].isnull().sum()}")
print(f"- Eksik 'Order_Date' sayısı (NaT): {df_sonuc['Order_Date'].isnull().sum()}")
print("- 'Order_Date' tipi:", df_sonuc['Order_Date'].dtype)
df_sonuc.head()

Temizleme sonrası kontrol:
- Eksik 'Category' sayısı: 0
- Eksik 'Quantity' sayısı: 0
- Eksik 'Price' sayısı: 0
- Eksik 'Order_Date' sayısı (NaT): 2
- 'Order_Date' tipi: datetime64[ns]


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total,islem_yili,islem_ayi
0,100,Customer_100,ORD-41285,2024-11-22,Blender,Home,3.0,38.000,Cash on Delivery,Shipped,114.00,2024.0,11.0
1,101,Customer_101,ORD-35783,2025-07-05,Smartphone,Electronics,2.0,542.195,PayPal,Processing,1084.39,2025.0,7.0
2,102,Customer_102,ORD-84355,2024-12-23,Tennis Racket,Sports,1.0,389.050,PayPal,Delivered,389.05,2024.0,12.0
3,103,Customer_103,ORD-57811,2025-03-19,Science,Books,5.0,233.920,PayPal,Processing,1169.60,2025.0,3.0
4,104,Customer_104,ORD-93614,2025-10-20,Biography,Books,1.0,552.510,Cash on Delivery,Processing,552.51,2025.0,10.0


## Özet - E-Ticaret Satış Verisi Temizliği Sunumu İçin

Bu notebook'ta, `messy_ecommerce_sales_data.csv` dosyasından alınan ham e-ticaret satış verisi üzerinde kapsamlı bir temizlik süreci gerçekleştirdik. Amacımız, veri analizi ve raporlama için güvenilir bir temel oluşturmaktı. Gerçekleşen başlıca veri sorunlarını ve bu sorunlara uygulanan çözümleri aşağıda bulabilirsiniz:

| Sorun | Tespit | Çözüm |
|---|---|---|
| Sütun adlarında boşluklar | `df.columns` | `.str.strip()` ile boşlukları temizleme |
| Eksik 'Category' değerleri | `df['Category'].isnull().sum()` | `fillna('Bilinmiyor')` ve `str.strip().str.title()` ile doldurma ve standardizasyon |
| 'Quantity' ve 'Price' sütunlarında sayısal olmayan değerler / eksikler | `df.info()`, `pd.to_numeric` hataları | `pd.to_numeric(errors='coerce')` ile sayısal tipe çevirme ve `median()` ile NaN'ları doldurma |
| 'Order_Date' string olarak saklı | `df.info()`, `df['Order_Date'].dtype` | `pd.to_datetime(errors='coerce')` ile datetime tipine çevirme |
| Tutarsız 'Total' tutarları | 'Total' ve 'Quantity * Price' karşılaştırması | `df['Total'] = df['Quantity'] * df['Price']` ile yeniden hesaplama |
| 'Payment_Method' boşluk tutarsızlıkları | `df['Payment_Method'].unique()` | `.str.strip()` ile boşlukları temizleme |
| Tekrar eden kayıtlar | `df.duplicated().sum()` | `df.drop_duplicates()` ile kaldırma |
| İndeks karışıklığı | — | `.reset_index(drop=True)` ile indeksi sıfırlama |

**Altın kural:** Ham veriyi asla doğrudan değiştirme; her zaman `.copy()` ile kopya üzerinde çalış, düzeltme kararlarını kod içinde açıkça belirt ve temizlenmiş veriyi ayrı bir dosyaya kaydet.